<a href="https://colab.research.google.com/github/Thangapandi1611/ml-safety-project/blob/main/week8/KNN_OOD_DECTIYION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import torch
import torch.nn as nn

from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix
import seaborn as sns
import pandas as pd
from PIL import Image
import os

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import matplotlib.pyplot as plt
import random
import numpy as np
import cv2

In [9]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
#LOADING PATHS
VEHICLE_MODEL_PATH="/content/drive/MyDrive/MLS/Vehicle_model.pth"
VALIDATION_PATH="/content/drive/MyDrive/MLS/validation"
TEST_PATH= "/content/drive/MyDrive/MLS/test"
FOG_PATH="/content/drive/MyDrive/MLS/test-fog"
NIGHT_PATH="/content/drive/MyDrive/MLS/test-night"

In [4]:
#LOAD VEHICLE MODEL
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = models.resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    1
)

model.load_state_dict(
    torch.load(
        VEHICLE_MODEL_PATH,
        map_location=device
    )
)

model.to(device)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [10]:
#Image Transform
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [5]:
#CREATE FEATURE EXTRACTOR
feature_extractor = nn.Sequential(
    *list(model.children())[:-1]
)

feature_extractor.to(device)

feature_extractor.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [11]:
class CarlaDataset(Dataset):

    def __init__(self, data_path, label_column, transform=None):

        self.data_path = data_path
        self.transform = transform

        self.labels = pd.read_csv(
            os.path.join(data_path, "labels.csv")
        )

        self.label_column = label_column

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        row = self.labels.iloc[idx]

        frame = str(row['frame']).zfill(6)

        img_path = os.path.join(
            self.data_path,
            "rgb-front",
            f"{frame}.jpg"
        )

        image = Image.open(img_path).convert("RGB")

        label = int(row[self.label_column])

        if self.transform:
            image = self.transform(image)

        return image, label

In [14]:
#CREATE DATASETS
train_dataset = CarlaDataset(
    VALIDATION_PATH,
    "has_vehicle",
    transform
)

id_dataset = CarlaDataset(
    TEST_PATH,
    "has_vehicle",
    transform
)

fog_dataset = CarlaDataset(
    FOG_PATH,
    "has_vehicle",
    transform
)

night_dataset = CarlaDataset(
    NIGHT_PATH,
    "has_vehicle",
    transform
)

In [12]:
#fEATURE EXTRACTION FUNCTION
def extract_features(
    dataset,
    feature_extractor,
    device
):

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False
    )

    features = []

    with torch.no_grad():

        for images, _ in loader:

            images = images.to(device)

            feat = feature_extractor(
                images
            )

            feat = feat.view(
                feat.size(0),
                -1
            )

            features.append(
                feat.cpu().numpy()
            )

    return np.vstack(features)

In [15]:
#extract features
train_features = extract_features(
    train_dataset,
    feature_extractor,
    device
)

id_features = extract_features(
    id_dataset,
    feature_extractor,
    device
)

fog_features = extract_features(
    fog_dataset,
    feature_extractor,
    device
)

night_features = extract_features(
    night_dataset,
    feature_extractor,
    device
)

In [16]:
#knn detector
knn = NearestNeighbors(
    n_neighbors=1
)

knn.fit(
    train_features
)

NearestNeighbors(n_neighbors=1)

In [17]:
#computing ood scores
id_distances, _ = knn.kneighbors(
    id_features
)

fog_distances, _ = knn.kneighbors(
    fog_features
)

night_distances, _ = knn.kneighbors(
    night_features
)

In [19]:
#Compute AUROC
#create labels
y_true = np.concatenate([
    np.zeros(len(id_distances)),
    np.ones(len(fog_distances)),
    np.ones(len(night_distances))
])
#Create score
ood_scores = np.concatenate([
    id_distances.flatten(),
    fog_distances.flatten(),
    night_distances.flatten()
])

In [21]:
#Compute AUROC
auroc_knn = roc_auc_score(
    y_true,
    ood_scores
)

print(
    "k-NN AUROC:",
    round(
        auroc_knn,
        4
    )
)

k-NN AUROC: 0.7799


In [22]:
print("MSP AUROC :", 0.8168)
print("k-NN AUROC:", auroc_knn)

MSP AUROC : 0.8168
k-NN AUROC: 0.7799389081790125


In [23]:
#FINDING WHICH SCENARIOS IS HARDER
#FOG
y_true_fog = np.concatenate([
    np.zeros(len(id_distances)),
    np.ones(len(fog_distances))
])

scores_fog = np.concatenate([
    id_distances.flatten(),
    fog_distances.flatten()
])

auroc_fog = roc_auc_score(
    y_true_fog,
    scores_fog
)

print(
    "Fog AUROC:",
    auroc_fog
)

Fog AUROC: 0.5973409336419753


In [24]:
#NIGHT
y_true_night = np.concatenate([
    np.zeros(len(id_distances)),
    np.ones(len(night_distances))
])

scores_night = np.concatenate([
    id_distances.flatten(),
    night_distances.flatten()
])

auroc_night = roc_auc_score(
    y_true_night,
    scores_night
)

print(
    "Night AUROC:",
    auroc_night
)

Night AUROC: 0.9625368827160494
